In [1]:
# Packages
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib import pyplot
import numpy as np
import pandas as pd
import seaborn
import sys
# Open flood_tool package
sys.path.append('../..')
import flood_tool as ft
# Open functions from geo.py
sys.path.append('..')
from geo import get_gps_lat_long_from_easting_northing
from geo import get_easting_northing_from_gps_lat_long

## Opening Datasets

In [2]:
# Open datasets
DF_typical_orig = pd.read_csv("../resources/example_data/typical_day.csv")
DF_wet_orig = pd.read_csv("../resources/example_data/wet_day.csv")
DF_rainfall = pd.read_csv("../resources/preprocessed_data/rainfall_stations.csv")
DF_level = pd.read_csv("../resources/preprocessed_data/level_stations_imputed.csv")
DF_wet_orig

,dateTime,stationReference,parameter,qualifier,unitName,value
0,2023-10-20T00:00:00Z,4205,level,Stage,mASD,0.137
1,2023-10-20T00:00:00Z,E6380,level,Stage,mAOD,2.146
2,2023-10-20T00:00:00Z,E7270,rainfall,Tipping Bucket Raingauge,mm,1.4
3,2023-10-20T00:00:00Z,2653,level,Stage,mASD,0.119
4,2023-10-20T00:00:00Z,3697TH,level,Downstream Stage,mASD,1.617
...,...,...,...,...,...,...
202207,2023-10-20T19:45:00Z,E72639,level,Tidal Level,mAOD,2.074
202208,2023-10-20T21:00:00Z,E72639,level,Tidal Level,mAOD,4.192
202209,2023-10-20T23:30:00Z,E72639,level,Tidal Level,mAOD,3.742
202210,2023-10-20T17:45:00Z,E72639,level,Tidal Level,mAOD,-1.543


In [3]:
DF_level

,stationReference,stationName,latitude,longitude,maxOnRecord,minOnRecord,typicalRangeHigh,typicalRangeLow,easting,northing
0,0130TH,Ewen,51.674733,-1.990400,0.945,-0.948,0.990,0.000,400760.571730,197329.368192
1,0144TH,Somerford Keynes,51.650724,-1.973061,1.484,0.325,1.600,0.363,401960.521598,194659.389917
2,0155TH,Oaksey,51.632205,-2.007169,1.887,-0.242,1.311,0.000,399600.530564,192599.408981
3,0184TH,Purton Stoke,51.613220,-1.874040,2.340,0.481,1.890,0.504,408818.519005,190495.374247
4,0190TH,Cricklade,51.646694,-1.865536,0.992,0.012,1.083,0.058,409400.521612,194219.366226
...,...,...,...,...,...,...,...,...,...,...
1113,E71839,Portsmouth,50.802280,-1.111170,2.066,0.050,1.250,0.151,462731.265440,100678.719642
1114,E71939,Bournemouth,50.714331,-1.874873,2.066,0.050,1.250,0.151,408930.528896,90529.727335
1115,E70739,Aberdeen,57.144060,-2.077360,2.066,0.050,1.250,0.151,395417.582934,805910.170180
1116,E74239,Tobermory,56.623110,-6.064220,2.066,0.050,1.250,0.151,150790.793963,755306.935288


## Finding Geographic Coordinates of Each Station and Data Cleaning

In [15]:
# Function to find latitude longitude and easting northing coordinates of each station in weather data DF
def Station_Finder(DF_orig, DF_rainfall, DF_level):
    DF = DF_orig.copy()
    lons = np.zeros(len(DF))
    lats = np.zeros(len(DF))
    easts = np.zeros(len(DF))
    norths = np.zeros(len(DF))
    values = np.zeros(len(DF))
# Find station using stationReference
    for i in range(len(DF)):
        station_ref = DF['stationReference'][i]
        if DF['parameter'][i] == 'rainfall':
            station_row = DF_rainfall[DF_rainfall['stationReference'] == station_ref].reset_index()
        elif DF['parameter'][i] == 'level':
            station_row = DF_level[DF_level['stationReference'] == station_ref].reset_index()
# Get coordinates if station can be found in database
        if len(station_row) > 0:
            lons[i] = station_row['longitude'][0]
            lats[i] = station_row['latitude'][0]
            easts[i] = station_row['easting'][0]
            norths[i] = station_row['northing'][0]
# np.nan if station cannot be found in database
        else:
            lons[i], lats[i], easts[i], norths[i] = np.nan, np.nan, np.nan, np.nan
#
# Clean values column
        try:
            values[i] = float(DF['value'][i])
        except:
            Value = DF['value'][i].split("|")
            values[i] = Value[0]
#
# Update columns
    DF['latitude'] = lats
    DF['longitude'] = lons
    DF['easting'] = easts
    DF['northing'] = norths
    DF['value'] = values
#
# Drop rows with no geographic coordinates or value found
    DF = DF.dropna(subset=['latitude', 'longitude', 'value']).reset_index().drop(columns='index')
    return (DF)

In [16]:
# Apply function to typical_day.csv
DF_typical = Station_Finder(DF_typical_orig, DF_rainfall, DF_level)
DF_typical

,dateTime,stationReference,parameter,qualifier,unitName,value,latitude,longitude,easting,northing
0,2021-10-10T00:00:00Z,000008,rainfall,Tipping Bucket Raingauge,mm,0.000,53.480556,-1.441674,437150.512108,398348.710960
1,2021-10-10T00:00:00Z,000028,rainfall,Tipping Bucket Raingauge,mm,0.000,53.500289,-1.673575,421750.534631,400448.642470
2,2021-10-10T00:00:00Z,000075TP,rainfall,Tipping Bucket Raingauge,mm,0.000,51.084022,-0.214597,525150.530744,133149.574480
3,2021-10-10T00:00:00Z,000076TP,rainfall,Tipping Bucket Raingauge,mm,0.000,51.701508,-0.747539,486650.531968,201049.405434
4,2021-10-10T00:00:00Z,000180TP,rainfall,Tipping Bucket Raingauge,mm,0.000,51.618838,0.173236,550550.534797,193349.379816
...,...,...,...,...,...,...,...,...,...,...
196001,2021-10-10T23:57:00Z,E8293,level,Stage,mASD,0.323,50.951995,0.074349,545800.569771,118999.673645
196002,2021-10-10T23:58:00Z,E1550,level,Stage,mASD,1.195,51.194615,0.278452,559300.558088,146399.623635
196003,2021-10-10T23:58:00Z,E8293,level,Stage,mASD,0.323,50.951995,0.074349,545800.569771,118999.673645
196004,2021-10-10T23:59:00Z,E1550,level,Stage,mASD,1.195,51.194615,0.278452,559300.558088,146399.623635


In [17]:
# Apply function to wet_day.csv
DF_wet = Station_Finder(DF_wet_orig, DF_rainfall, DF_level)
DF_wet

,dateTime,stationReference,parameter,qualifier,unitName,value,latitude,longitude,easting,northing
0,2023-10-20T00:00:00Z,4205,level,Stage,mASD,0.137,52.765357,-1.214327,453110.521317,318928.955875
1,2023-10-20T00:00:00Z,E6380,level,Stage,mAOD,2.146,50.989632,0.755050,593450.561502,124749.623244
2,2023-10-20T00:00:00Z,E7270,rainfall,Tipping Bucket Raingauge,mm,1.400,50.890762,0.554388,579750.521629,113249.683179
3,2023-10-20T00:00:00Z,2653,level,Stage,mASD,0.119,51.885940,-2.093645,393651.531128,220824.348069
4,2023-10-20T00:00:00Z,3697TH,level,Downstream Stage,mASD,1.617,51.465414,-0.322425,516629.567145,175383.521802
...,...,...,...,...,...,...,...,...,...,...
200255,2023-10-20T19:45:00Z,E72639,level,Tidal Level,mAOD,2.074,51.499990,-2.728468,349530.519072,178146.472275
200256,2023-10-20T21:00:00Z,E72639,level,Tidal Level,mAOD,4.192,51.499990,-2.728468,349530.519072,178146.472275
200257,2023-10-20T23:30:00Z,E72639,level,Tidal Level,mAOD,3.742,51.499990,-2.728468,349530.519072,178146.472275
200258,2023-10-20T17:45:00Z,E72639,level,Tidal Level,mAOD,-1.543,51.499990,-2.728468,349530.519072,178146.472275


In [13]:
# Rows where value column is not a single numeric value
num = pd.to_numeric(DF_wet_orig["value"], errors='coerce')
failed_rows = DF_wet_orig[num.isna & DF_wet_orig.notna()]
failed_rows

,dateTime,stationReference,parameter,qualifier,unitName,value
72,2023-10-20T00:00:00Z,4710,level,Stage,mASD,0.850|0.847
231,2023-10-20T00:00:00Z,4187,level,Stage,mASD,0.101|0.098
318,2023-10-20T00:00:00Z,4161,level,Stage,mASD,1.538|1.230
1766,2023-10-20T00:15:00Z,4710,level,Stage,mASD,0.850|0.846
1924,2023-10-20T00:15:00Z,4187,level,Stage,mASD,0.100|0.097
...,...,...,...,...,...,...
189558,2023-10-20T23:15:00Z,4427,level,Stage,mASD,2.237|2.328
190645,2023-10-20T23:15:00Z,2180TH,level,Stage,mASD,1.450|1.446
191297,2023-10-20T23:30:00Z,4427,level,Stage,mASD,2.355|2.249
193155,2023-10-20T23:45:00Z,4427,level,Stage,mASD,2.251|2.359


In [18]:
# Verify all rows in value column are now a single numeric value
num = pd.to_numeric(DF_wet["value"], errors='coerce')
failed_rows = DF_wet[num.isna()]
failed_rows

,dateTime,stationReference,parameter,qualifier,unitName,value,latitude,longitude,easting,northing


## Outputting Processed Data

In [7]:
# Define output directory
Output_Diri = '../resources/preprocessed_data/example_data_preprocessed/'

In [8]:
# Output preprocessed typical day data as new csv file
DF_typical.to_csv(Output_Diri+'typical_day_preprocessed.csv', index=False)

In [9]:
# Output preprocessed wet day data as new csv file
DF_wet.to_csv(Output_Diri+'wet_day_preprocessed.csv', index=False)